# SPX calibration — Legendre chaos

Fit one chaos model with two Brownian drivers, order 3, and 10 Legendre functions per driver. All maturities use Monte Carlo pricing with exact first- and second-moment control variates.

The configuration below controls initialization, sampling budgets, optimizer settings, stopping, and GPU selection. Both regularization weights are zero.

Use **Restart Kernel → Run All**. All quotes are used for fitting; independent Monte Carlo samples evaluate the fitted market surface.


SPX inputs and numerical results are not distributed. Supply your own data as described in the repository README.


## 1. Imports and market data

Load the same SPX dataset as the piecewise constant notebook. Each maturity retains
its own rate and dividend yield. Paths work from the project root or this directory.


In [ ]:
from pathlib import Path
import hashlib
import importlib
import json
import pickle
import random
import sys
import time

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import torch
from IPython.display import Markdown, display

ROOT = next(
    (path for path in (Path.cwd(), *Path.cwd().parents)
     if (path / "wiener_chaos/core/model.py").exists()),
    None,
)
if ROOT is None:
    raise FileNotFoundError("Open this notebook from within the project directory.")
STUDY_DIR = ROOT / "Real data" / "Legendre"
NOTEBOOK_PATH = STUDY_DIR / "Calibration.ipynb"
if str(ROOT) in sys.path:
    sys.path.remove(str(ROOT))
sys.path.insert(0, str(ROOT))

from wiener_chaos.experiments.calibration import save_artifacts
from wiener_chaos.experiments.provenance import source_fingerprint
from wiener_chaos.pricing.options import compute_ivs
from wiener_chaos.core import chaos as general_chaos_module
general_chaos_module = importlib.reload(general_chaos_module)
from wiener_chaos.pricing import heston as heston_module
heston_module = importlib.reload(heston_module)
from wiener_chaos.core import model as pricing_module
pricing_module = importlib.reload(pricing_module)
ModelStock = pricing_module.ModelStock
from wiener_chaos.experiments import calibration as timing_module
timing_module = importlib.reload(timing_module)
print(f"Pricing module loaded: {pricing_module.__file__}")

DATA_PATH = ROOT / "Real data" / "real_data.pkl"
if not DATA_PATH.is_file():
    raise FileNotFoundError(
        "SPX inputs are private and are not distributed. Supply your own quotes "
        "and run Real data/Data.ipynb first; see the README input format."
    )
with DATA_PATH.open("rb") as handle:
    data = pickle.load(handle)

pd.set_option("display.max_columns", 12)
plt.rcParams.update({"figure.figsize": (10, 4), "axes.grid": True, "grid.alpha": 0.25})

## 2. Calibration settings

The configuration defines the model order, basis size, sampling budgets,
optimizer, stopping criteria, and device. The monitored objective is
vega-weighted price MSE; independently evaluated IV errors are reported after
fitting. A run reaching the iteration cap is flagged for convergence review.

Set the per-maturity pricing allocations to match your own dataset. Training
and comparison draws use separate seeds. Adjust `CUDA_DEVICE_INDEX` for your
machine; CPU execution is also supported.

### Heston initialization

Keep `CONFIG["initialization"]["fit_heston"] = True` for a fitted-Heston warm
start. Set it to `False` to use the surface heuristic without Heston
optimization. Projection and chaos calibration still run with the configured
budgets. Use the same choice in the individual and study notebooks.


In [ ]:
CUDA_DEVICE_INDEX = 2
if torch.cuda.is_available() and not 0 <= CUDA_DEVICE_INDEX < torch.cuda.device_count():
    raise RuntimeError(f"CUDA device {CUDA_DEVICE_INDEX} is unavailable.")
DEVICE = f"cuda:{CUDA_DEVICE_INDEX}" if torch.cuda.is_available() else "cpu"
if DEVICE == "cpu":
    torch.set_num_threads(1)

N_MATURITIES = len(data["maturities"])
CONFIG = {
    # Master seed for reproducible initialization and sampling.
    "base_seed": 42,
    # Device used for model tensors and pricing.
    "device": DEVICE,
    # Algorithm for evaluating conditional chaos moments.
    "backend": "tensor",
    # Number of Brownian drivers.
    "d": 2,
    # Maximum total chaos order.
    "P": 3,
    # Legendre basis: ten time functions per Brownian driver.
    "basis": "legendre",
    "M": 10,
    "base_jitter": 1e-12,
    # Paths evaluated per chunk by the general-basis pricing kernel.
    "pricing_chunk_size": 32_768,
    "init_scale": 1e-3,
    # How the starting chaos coefficients are obtained.
    "initialization": {
        # Choose a fitted-Heston projection or a random chaos start.
        "type": "calibrate_heston",
        # Starting point for the Heston parameter optimizer.
        "initial_guess": "surface",
        # False uses the surface heuristic directly, without Heston optimization.
        "fit_heston": True,
        # General-basis projection currently supports the plain estimator.
        "projection_control": "none",
        # Antithetic projection is currently supported only for piecewise bases.
        "projection_antithetic": False,
        # Independent projection estimates averaged before normalization.
        "projection_repetitions": 1,
        # Brownian paths in each terminal-factor projection batch.
        "N": 5_000,
        # Number of projection batches; total paths are N times Ns.
        "Ns": 128,
        # Number of time-grid points in each Heston simulation.
        "n_steps_heston": 201,
        # Settings passed to the Heston optimizer.
        "options": {
            # Maximum Heston optimizer iterations.
            "maxiter": 500,
            # Heston optimizer tolerance for objective improvement.
            "ftol": 1e-12,
            # Heston optimizer tolerance for the gradient.
            "gtol": 1e-9,
        },
    },
    # Pricing allocations at calibration maturities.
    "pricing_calibr": {
        # Integration method for each calibration maturity (MC or Gauss-Hermite).
        "method": ["MC"] * 10,
        # MC paths per calibration maturity; None for quadrature.
        "N": [400_000] * 4 + [200_000] * 6,
        # Independent MC batches per calibration maturity.
        "Ns": [1] * 10,
        # Gauss-Hermite nodes per dimension; None for MC.
        "nodes": [None] * 10,
    },
    # MC paths per maturity for fixed validation and evaluation samples.
    "comparison_N_per_mat": [200_000] * 4 + [100_000] * 6,
    # Optimizer, control-variate, and stopping settings.
    "calibration": {
        # Maximum number of optimizer steps.
        "n_iter": 600,
        # Initial AdamW learning rate.
        "lr": 1e-4,
        # Highest stock-price power used as an MC control variate.
        "poly_cv_degree": 2,
        # Non-improving monitoring checks allowed before early stopping.
        "patience": 60,
        # Minimum monitored improvement needed to reset patience.
        "tol_error": 2e-8,
        # Optimizer steps before activating MC control variates.
        "warmup_without_cv": 0,
        # Best monitored loss must be below this to stop early.
        "limit_error_above": 2e-6,
        # Steps between fresh training MC draws; None keeps draws fixed.
        "simulate_every": 50,
        # AdamW decoupled weight-decay coefficient.
        "reg_lambda": 0,
        # Weight of the penalty toward the starting chaos coefficients.
        "anchor_lambda": 0,
        # MC paths reserved to estimate control-variate weights.
        #"n_samples_cv": 10_000,
        # Steps between fixed validation checks; None monitors training loss.
        "check_validation_set_MC": None,
        # Steps between price-loss and speed reports; None suppresses reports.
        "print_loss_every_it": 25,
        # Whether model.calibrate displays fit plots.
        "print_plots": False,
        # Learning-rate scheduler settings for stalled monitoring loss.
        "plateau_kwargs": {
            # Whether lower or higher monitored values are better.
            "mode": "min",
            # Multiplier applied to the learning rate on a plateau.
            "factor": 0.95,
            # Non-improving checks before reducing the learning rate.
            "patience": 15,
            # Minimum change counted as scheduler improvement.
            "threshold": 1e-7,
            # Interpret threshold as absolute or relative.
            "threshold_mode": "abs",
            # Checks to wait after a learning-rate reduction.
            "cooldown": 0,
            # Lowest learning rate allowed by the scheduler.
            "min_lr": 1e-6,
            # Smallest learning-rate change worth applying.
            "eps": 1e-8,
        },
    },
}
OUTPUT_DIR = STUDY_DIR / "Statistics calibration" / "single"
FIXED_VALIDATION_SEED = 4_000_042
FIXED_EVALUATION_SEED = 5_000_042

for key in ("strikes", "prices", "iv", "iv_bid", "iv_ask", "r_s", "q_s"):
    if len(data[key]) != N_MATURITIES:
        raise ValueError(f"{key} must contain one entry per maturity.")
for rows in zip(data["strikes"], data["prices"], data["iv"], data["iv_bid"], data["iv_ask"]):
    if len({len(row) for row in rows}) != 1:
        raise ValueError("Quote rows must have aligned strikes, prices, and IVs.")
for key in ("method", "N", "Ns", "nodes"):
    if len(CONFIG["pricing_calibr"][key]) != N_MATURITIES:
        raise ValueError(f"Pricing {key} must contain one entry per maturity.")

if len(CONFIG["comparison_N_per_mat"]) != N_MATURITIES:
    raise ValueError("comparison_N_per_mat must contain one entry per maturity.")

print(f"Device: {DEVICE} | Basis: {CONFIG['basis']} | Order: {CONFIG['P']} | "
      f"Functions per driver: {CONFIG['M']} | Initialization: {CONFIG['initialization']['type']}")
print(f"Training paths: {sum(CONFIG['pricing_calibr']['N']):,} | "
      f"Independent evaluation paths: {sum(CONFIG['comparison_N_per_mat']):,} | "
      f"Pricing chunk size: {CONFIG['pricing_chunk_size']:,}")
print(f"Results: {OUTPUT_DIR}")
display(pd.DataFrame({
    "maturity": data["maturities"],
    "quotes": [len(row) for row in data["strikes"]],
    "rate": data["r_s"],
    "dividend_yield": data["q_s"],
    "pricing": CONFIG["pricing_calibr"]["method"],
    "training_N": CONFIG["pricing_calibr"]["N"],
    "validation_evaluation_N": CONFIG["comparison_N_per_mat"],
}))


## 3. Calibration routines

These cells define the workflow helpers. They do not start calibration.

### Reproducibility, saving, and error metrics

In [ ]:
def seed_everything(seed):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)


def json_safe(value):
    """Detach nested tensors and use JSON null for nonfinite diagnostics."""
    if torch.is_tensor(value):
        return json_safe(value.detach().cpu().tolist())
    if isinstance(value, np.ndarray):
        return json_safe(value.tolist())
    if isinstance(value, dict):
        return {str(k): json_safe(v) for k, v in value.items()}
    if isinstance(value, (list, tuple)):
        return [json_safe(v) for v in value]
    if isinstance(value, (float, np.floating)):
        return float(value) if np.isfinite(value) else None
    if isinstance(value, np.integer):
        return int(value)
    return value


def save_json(path, payload):
    path = Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)
    temporary = path.with_suffix(path.suffix + ".tmp")
    temporary.write_text(json.dumps(json_safe(payload), indent=2, allow_nan=False) + "\n")
    temporary.replace(path)


def surface_metrics(actual, expected):
    """Do not silently discard invalid IVs or mismatched surface entries."""
    if len(actual) != len(expected):
        raise ValueError("Different numbers of maturity rows.")
    errors = []
    invalid = 0
    for row, reference in zip(actual, expected):
        if len(row) != len(reference):
            raise ValueError("Different numbers of strikes.")
        for value, target in zip(row, reference):
            if value is None or not np.isfinite(value) or not np.isfinite(target):
                invalid += 1
            else:
                errors.append(abs(float(value) - float(target)))
    return {
        "mae": float(np.mean(errors)) if errors and not invalid else None,
        "invalid_quotes": invalid,
    }

### General-basis model and independent evaluation

Starting and fitted chaos models share independent comparison draws. Seeded
comparison sampling does not advance the training random stream.


In [ ]:
def initialize_parameters_timed(model, *, timing=None, **kwargs):
    if model.device.type == "cuda":
        torch.cuda.synchronize(model.device)
    started = time.perf_counter()
    if timing is not None:
        timing.start_initialization()
        kwargs["timing_check"] = timing.check_initialization
    initial = model.initialize_params(**kwargs)
    if model.device.type == "cuda":
        torch.cuda.synchronize(model.device)
    seconds = time.perf_counter() - started
    if timing is not None:
        timing.finish_initialization(seconds)
    return initial, seconds


def build_model(data, config):
    model = ModelStock(data["S0"], torch.device(config["device"]), torch.float32)
    model.initialize_chaos(
        config["basis"], max(data["maturities"]), config["d"], config["P"], config["M"],
        base_jitter=config["base_jitter"], backend=config["backend"],
    )
    model.chaos.chunk_size = int(config["pricing_chunk_size"])
    if model.chaos.chunk_size < 1:
        raise ValueError("pricing_chunk_size must be positive.")
    model.set_up_for_Ts(data["maturities"], data["r_s"], data["q_s"])
    # Seeded validation and final evaluation use independent comparison samples.
    comparison_N = {round(float(T), 6): count for T, count in zip(
        data["maturities"], config["comparison_N_per_mat"],
    )}
    original_seeded_sample = model._simulate_X_block_with_seed

    def fixed_comparison_sample(maturities_block, method_block, N_block, Ns_block, seed=None):
        if seed is not None:
            N_block = [comparison_N[round(float(T), 6)] for T in maturities_block]
        return original_seeded_sample(
            maturities_block, method_block, N_block, Ns_block, seed,
        )

    model._simulate_X_block_with_seed = fixed_comparison_sample
    return model


@torch.no_grad()
def evaluate_surface(model, coefficients, data, pricing, calibration, seed):
    draws = model._simulate_X_block_with_seed(
        data["maturities"], pricing["method"], pricing["N"], pricing["Ns"], seed,
    )
    prices = model._prices_model_block(
        coefficients, draws, data["maturities"], data["strikes"],
        pricing["method"], pricing["nodes"],
        poly_cv_degree=calibration["poly_cv_degree"],
        n_samples_cv=calibration.get("n_samples_cv"),
    )
    prices = [row.detach().cpu().tolist() for row in prices]
    ivs = compute_ivs(prices, data["maturities"], data["strikes"],
                      data["S0"], data["r_s"], data["q_s"])
    return prices, ivs


def fitted_heston_surface(params, data):
    prices = [
        heston_module.heston_price_vec(params, r, q, T, data["S0"], strikes).tolist()
        for T, strikes, r, q in zip(data["maturities"], data["strikes"], data["r_s"], data["q_s"])
    ]
    ivs = compute_ivs(prices, data["maturities"], data["strikes"],
                      data["S0"], data["r_s"], data["q_s"])
    return prices, ivs


## 4. Run one calibration

Fit Heston to the market quotes and project its terminal factor, or select random
initialization. Preserve the starting coefficients before calibration modifies
them. Evaluate the start, calibrate on the market prices, and save the result
before plotting. Training, validation, and final evaluation have separate seeds.

The Heston optimizer status is displayed and saved. A run reaching the chaos
iteration limit should be inspected before drawing conclusions about convergence.

This run also saves initialization time and 25-iteration timing windows in
`result.json`. The windows use the exact timestamps of the printed iterations/s,
after setup and after completed optimizer steps. Run it once under normal
conditions before starting the study.


In [ ]:
progress = display(Markdown("Initializing the SPX Legendre model…"), display_id=True)
seed = CONFIG["base_seed"]
calibration_seed = seed + 3_000_000
seed_everything(seed)
model = build_model(data, CONFIG)
pricing = CONFIG["pricing_calibr"]
init_config = CONFIG["initialization"]
# Record a healthy run for the study's timing checks; no limits apply here.
timing = timing_module.CalibrationTimer(simulate_every=CONFIG["calibration"]["simulate_every"])
fit = None
prices_heston = iv_heston = None
if init_config["type"] == "calibrate_heston":
    initial, init_seconds = initialize_parameters_timed(
        model, timing=timing, type="calibrate_heston", maturities=data["maturities"],
        strikes=data["strikes"], prices_market=data["prices"],
        r=data["r_s"], q=data["q_s"], seed=seed,
        **{key: value for key, value in init_config.items() if key != "type"},
    )
    fit = model.heston_fit_result
    if fit is not None:
        print(f"Heston optimizer success: {fit.success}; iterations: {fit.nit}; "
              f"objective: {fit.fun:.6e}; {fit.message}")
    else:
        print("Using heuristic Heston parameters; optimizer skipped.")
    print(f"Heston parameters (kappa, theta, sigma, rho, v0): {model.heston_params}")
    prices_heston, iv_heston = fitted_heston_surface(model.heston_params, data)
elif init_config["type"] == "random":
    initial, init_seconds = initialize_parameters_timed(
        model, timing=timing, type="random", scale=CONFIG["init_scale"],
    )
else:
    raise ValueError(f"Unknown initialization type: {init_config['type']}")
initial_coefficients = initial.detach().cpu().clone()
progress.update(Markdown("Evaluating the starting chaos model…"))
prices_initial, iv_initial = evaluate_surface(
    model, initial, data, pricing, CONFIG["calibration"], FIXED_EVALUATION_SEED,
)

progress.update(Markdown("Calibrating the Legendre coefficients…"))
seed_everything(calibration_seed)
d_a, prices_model, loss_ins, loss_oos, fit_seconds = model.calibrate(
    data["iv"], data["prices"], [data["maturities"]], data["strikes"],
    pricing["method"], pricing["N"], pricing["Ns"], pricing["nodes"],
    d_a=initial, validation_seed=FIXED_VALIDATION_SEED,
    evaluation_seed=FIXED_EVALUATION_SEED, timing_callback=timing.training_step,
    **CONFIG["calibration"],
)
if not torch.isfinite(d_a).all():
    raise FloatingPointError("Nonfinite calibrated coefficients.")
iv_model = compute_ivs(prices_model, data["maturities"], data["strikes"],
                       data["S0"], data["r_s"], data["q_s"])
monitor_name = ("best_training_price_mse" if CONFIG["calibration"]["check_validation_set_MC"] is None
                else "best_validation_price_mse")
result = json_safe({
    "config": CONFIG, "seed": seed, "projection_seed": seed,
    "calibration_seed": calibration_seed,
    "validation_seed": (FIXED_VALIDATION_SEED
                        if CONFIG["calibration"]["check_validation_set_MC"] is not None else None),
    "evaluation_seed": FIXED_EVALUATION_SEED,
    "training_N_per_mat": pricing["N"],
    "comparison_N_per_mat": CONFIG["comparison_N_per_mat"],
    "d_a": d_a, "initial_coefficients": initial_coefficients,
    "heston_init_params": getattr(model, "heston_params", None),
    "heston_initial_guess": getattr(model, "heston_initial_guess", None),
    "heston_fit": ({"success": bool(fit.success), "iterations": int(fit.nit),
                    "objective": float(fit.fun), "message": str(fit.message)} if fit is not None else None),
    "maturities": data["maturities"], "r_s": data["r_s"], "q_s": data["q_s"],
    "loss_ins": loss_ins, "loss_oos": loss_oos,
    "calibration_iterations": model.last_calibration_iterations,
    monitor_name: model.last_calibration_monitor_loss,
    "prices_heston": prices_heston, "iv_heston": iv_heston,
    "prices_initial": prices_initial, "iv_initial": iv_initial,
    "prices_model": prices_model, "iv_model": iv_model,
    "iv_heston_metrics": surface_metrics(iv_heston, data["iv"]) if iv_heston is not None else None,
    "iv_initial_metrics": surface_metrics(iv_initial, data["iv"]),
    "iv_calibration_metrics": surface_metrics(iv_model, data["iv"]),
    "timing_profile": timing.profile(),
    "elapsed_seconds_init": init_seconds,
    "elapsed_seconds_chaos_calibration": fit_seconds,
    "elapsed_seconds_total": init_seconds + fit_seconds,
    "data_sha256": hashlib.sha256(DATA_PATH.read_bytes()).hexdigest(),
    "source_sha256": {name: source_fingerprint(ROOT / name)
                      for name in ("wiener_chaos/core/model.py", "wiener_chaos/pricing/heston.py", "wiener_chaos/core/chaos.py",
                                   "wiener_chaos/core/basis.py", "wiener_chaos/core/basis_integrals.py", "wiener_chaos/core/utils.py", "wiener_chaos/pricing/options.py",
                                   "wiener_chaos/experiments/calibration.py", "wiener_chaos/experiments/provenance.py")},
    "notebook_source_sha256": hashlib.sha256(NOTEBOOK_PATH.read_bytes()).hexdigest(),
    "runtime": {"torch": torch.__version__, "numpy": np.__version__, "device": CONFIG["device"]},
})
save_json(OUTPUT_DIR / "result.json", result)
mae = result["iv_calibration_metrics"]["mae"]
mae_text = f"{mae * 1e4:.2f} bp" if mae is not None else "unavailable (invalid IVs)"
progress.update(Markdown(
    f"Initialization: {init_seconds:.2f}s | Calibration: {fit_seconds:.2f}s | "
    f"Iterations: {model.last_calibration_iterations} | SPX IV MAE: {mae_text}"
))
if model.last_calibration_iterations >= CONFIG["calibration"]["n_iter"]:
    print("The iteration cap was reached; inspect convergence before interpreting this fit.")


## 5. Fit quality and runtime

Compare the fitted Heston model, projected starting model, and final chaos model.
IV errors are in basis points (`1 bp = 0.0001`); the bid/ask percentage counts quotes
whose fitted IV is inside the supplied market interval. Invalid IVs invalidate MAE.

Starting and final chaos prices use common independent Monte Carlo draws. The
Heston baseline uses the same Fourier grid as the Heston fitting objective.
Initialization time includes the Heston fit and projection. Calibration time
includes training and final pricing within `model.calibrate()`. The displayed sum
excludes setup, baseline evaluations, summary tables, and plots.


In [ ]:
heston_label = "Fitted Heston" if fit is not None else "Heuristic Heston"
comparison_surfaces = {}
if iv_heston is not None:
    comparison_surfaces[heston_label] = iv_heston
start_label = "Projected Heston start" if init_config["type"] == "calibrate_heston" else "Random start"
comparison_surfaces[start_label] = iv_initial
comparison_surfaces["Calibrated Legendre"] = iv_model

fit_rows = []
summary_rows = []
for label, fitted_surface in comparison_surfaces.items():
    inside_total = 0
    for T, fitted, target, bid, ask in zip(
        data["maturities"], fitted_surface, data["iv"], data["iv_bid"], data["iv_ask"],
    ):
        metrics = surface_metrics([fitted], [target])
        values = np.asarray(fitted, dtype=float)
        inside = np.isfinite(values) & (values >= np.asarray(bid)) & (values <= np.asarray(ask))
        inside_total += int(inside.sum())
        fit_rows.append({
            "model": label, "maturity": T, "quotes": len(values),
            "iv_mae_bp": None if metrics["mae"] is None else 1e4 * metrics["mae"],
            "invalid_iv_quotes": metrics["invalid_quotes"],
            "within_bid_ask_pct": 100 * inside.mean(),
        })
    metrics = surface_metrics(fitted_surface, data["iv"])
    n_quotes = sum(len(row) for row in fitted_surface)
    summary_rows.append({
        "model": label, "quotes": n_quotes,
        "iv_mae_bp": None if metrics["mae"] is None else 1e4 * metrics["mae"],
        "invalid_iv_quotes": metrics["invalid_quotes"],
        "within_bid_ask_pct": 100 * inside_total / n_quotes,
    })
fit_summary = pd.DataFrame(summary_rows).set_index("model")
fit_by_maturity = pd.DataFrame(fit_rows).set_index(["model", "maturity"])
display(fit_summary.round(4))
display(fit_by_maturity.round(4))
print(f"Initialization: {init_seconds:.2f}s | Calibration: {fit_seconds:.2f}s | "
      f"Combined: {init_seconds + fit_seconds:.2f}s | Iterations: {model.last_calibration_iterations}")


In [ ]:
def plot_spx_smiles(data, surfaces):
    columns = 3
    rows = (len(data["maturities"]) + columns - 1) // columns
    fig, axes = plt.subplots(rows, columns, figsize=(14, 3.4 * rows), squeeze=False)
    styles = {
        "Fitted Heston": {"linestyle": "--", "color": "tab:orange"},
        "Heuristic Heston": {"linestyle": "--", "color": "tab:orange"},
        "Projected Heston start": {"linestyle": ":", "color": "tab:green"},
        "Random start": {"linestyle": ":", "color": "tab:green"},
        "Calibrated Legendre": {"color": "tab:red"},
    }
    for index, (ax, T, strikes, market, bid, ask, r, q) in enumerate(zip(
        axes.flat, data["maturities"], data["strikes"], data["iv"],
        data["iv_bid"], data["iv_ask"], data["r_s"], data["q_s"],
    )):
        forward = data["S0"] * np.exp((r - q) * T)
        x = np.log(np.asarray(strikes) / forward)
        ax.fill_between(x, 100 * np.asarray(bid), 100 * np.asarray(ask),
                        alpha=.25, label="Market bid/ask")
        ax.plot(x, 100 * np.asarray(market), ".", color="tab:blue", label="Market IV")
        for label, surface in surfaces.items():
            ax.plot(x, 100 * np.asarray(surface[index], dtype=float),
                    label=label, **styles[label])
        ax.set(title=f"T = {T:g}", xlabel="log(K / forward)", ylabel="Implied volatility (%)")
    for ax in list(axes.flat)[len(data["maturities"]):]:
        ax.set_visible(False)
    handles, labels = axes.flat[0].get_legend_handles_labels()
    fig.legend(handles, labels, loc="upper center", ncol=3, bbox_to_anchor=(.5, 1.01))
    fig.suptitle("SPX — Heston initialization and calibrated Legendre", y=1.04)
    fig.tight_layout()
    plt.show()
    return fig

figures = {"calibration": plot_spx_smiles(data, comparison_surfaces)}


## 6. Saved results

`Statistics calibration/single/result.json` records the configuration, fitted
coefficients, prices, implied volatilities, seeds, timings, and provenance.
`coefficients/` contains `initial_coefficients.pt` and `calibrated_coefficients.pt`;
`tables/` contains `fit_summary.csv` and `fit_by_maturity.csv`. Comparison figures
are saved in `figures/` as PNG and PDF files.


In [ ]:
save_artifacts(
    OUTPUT_DIR,
    coefficients={"initial_coefficients": result["initial_coefficients"],
                  "calibrated_coefficients": result["d_a"]},
    tables={"fit_summary": fit_summary, "fit_by_maturity": fit_by_maturity},
    figures=figures,
)
print(f"Single-calibration results saved in: {OUTPUT_DIR}")
